In [1]:
#Q4
import random
#part a
def func(val):
    return -val**2 + 14*val + 5

def bin_to_int(bits):
    return int("".join(map(str, bits)), 2)

def eval_fit(bits):
    return func(bin_to_int(bits))

def select_roulette(group):
    fits = [eval_fit(item) for item in group]
    total = sum(fits)
    if total == 0:
        return random.choice(group)
    r = random.random()
    acc = 0
    for item, fval in zip(group, fits):
        acc += fval / total
        if r <= acc:
            return item

def crossover_one(p1, p2, idx):
    c1 = p1[:idx] + p2[idx:]
    c2 = p2[:idx] + p1[idx:]
    return c1, c2

def flip_bits(bits, prob):
    return [1 - b if random.random() < prob else b for b in bits]

samples = [[0,1,1,0], [1,0,0,1], [1,1,0,0], [0,0,1,1]]
print("Testing decode and fitness:\n")
for s in samples:
    val = bin_to_int(s)
    fit_val = eval_fit(s)
    print(f"{s}-> x = {val}, f(x) = {fit_val}")

Testing decode and fitness:

[0, 1, 1, 0]-> x = 6, f(x) = 53
[1, 0, 0, 1]-> x = 9, f(x) = 50
[1, 1, 0, 0]-> x = 12, f(x) = 29
[0, 0, 1, 1]-> x = 3, f(x) = 38


In [2]:
#part b and c
import random, statistics

def execute_ga(size, gens, mut_prob, use_elite):
    pop = [[random.randint(0,1) for _ in range(4)] for _ in range(size)]
    best_val_seen = float('-inf')
    reached_opt = False
    gen_hit_50 = None

    for g in range(gens):
        fit_list = [eval_fit(ind) for ind in pop]
        decoded_vals = [bin_to_int(ind) for ind in pop]

        max_fit = max(fit_list)
        max_x = decoded_vals[fit_list.index(max_fit)]

        if max_fit > best_val_seen:
            best_val_seen = max_fit

        if max_x == 7:
            reached_opt = True

        if gen_hit_50 is None and max_fit >= 50:
            gen_hit_50 = g

        next_pop = []

        if use_elite:
            elite_ind = pop[fit_list.index(max_fit)]
            next_pop.append(elite_ind)

        while len(next_pop) < size:
            parent_a = select_roulette(pop)
            parent_b = select_roulette(pop)

            cut = random.randint(1, 3)
            child_a, child_b = crossover_one(parent_a, parent_b, cut)

            child_a = flip_bits(child_a, mut_prob)
            child_b = flip_bits(child_b, mut_prob)

            next_pop.append(child_a)
            if len(next_pop) < size:
                next_pop.append(child_b)

        pop = next_pop

    return best_val_seen, reached_opt, gen_hit_50


def test_elitism_effect():
    runs = 30

    for elite_flag in [False, True]:
        best_vals = []
        opt_count = 0
        gens_50 = []

        for _ in range(runs):
            best_v, found_flag, g50 = execute_ga(4, 20, 0.1, elite_flag)

            best_vals.append(best_v)
            if found_flag:
                opt_count += 1
            if g50 is not None:
                gens_50.append(g50)

        print(f"\nElitism = {elite_flag}")
        print("Average Best Fitness:", sum(best_vals)/runs)
        print("Runs reaching x=7:", opt_count)
        print("Avg generations to f>=50:",
              sum(gens_50)/len(gens_50) if gens_50 else "N/A")


def test_mutation_rates():
    runs = 30
    probs = [0.01, 0.1, 0.3, 0.5]

    print("\nMutation Rate vs Avg Best Fitness")
    print("----------------------------------")

    for mp in probs:
        best_vals = []

        for _ in range(runs):
            best_v, _, _ = execute_ga(4, 20, mp, False)
            best_vals.append(best_v)

        print(f"p_m = {mp}-> Avg Best Fitness = {sum(best_vals)/runs}")


test_elitism_effect()
test_mutation_rates()


Elitism = False
Average Best Fitness: 53.766666666666666
Runs reaching x=7: 23
Avg generations to f>=50: 0.3333333333333333

Elitism = True
Average Best Fitness: 53.7
Runs reaching x=7: 21
Avg generations to f>=50: 0.4

Mutation Rate vs Avg Best Fitness
----------------------------------
p_m = 0.01-> Avg Best Fitness = 52.93333333333333
p_m = 0.1-> Avg Best Fitness = 53.766666666666666
p_m = 0.3-> Avg Best Fitness = 53.96666666666667
p_m = 0.5-> Avg Best Fitness = 54.0


In [6]:
#Q5
#part a
import random

def create_individual():
    return [(random.randint(0,2), random.randint(0,3)) for _ in range(6)]

def calc_conflicts(individual):
    clash = 0
    for a in range(len(individual)):
        for b in range(a+1, len(individual)):
            if individual[a] == individual[b]:
                clash += 1
    return clash

def compute_score(individual):
    return 100 - (10 * calc_conflicts(individual))

print("Chromosome\t\t\t\t\t\tConflicts\tFitness")
for _ in range(5):
    indiv = create_individual()
    clashes = calc_conflicts(indiv)
    score = compute_score(indiv)
    print(f"{indiv}\t{clashes}\t\t{score}")

Chromosome						Conflicts	Fitness
[(2, 0), (0, 2), (2, 2), (2, 3), (0, 3), (1, 2)]	0		100
[(0, 2), (2, 1), (0, 0), (2, 1), (1, 2), (1, 3)]	1		90
[(0, 3), (0, 3), (2, 0), (2, 1), (1, 0), (2, 3)]	1		90
[(1, 3), (0, 1), (0, 1), (0, 3), (0, 1), (1, 0)]	3		70
[(1, 3), (0, 3), (1, 2), (1, 1), (1, 1), (2, 2)]	1		90


In [7]:
#part b
def mix_parents(parent_a, parent_b, cut_point):
    child_a = parent_a[:cut_point] + parent_b[cut_point:]
    child_b = parent_b[:cut_point] + parent_a[cut_point:]
    return child_a, child_b

def fix_duplicates(indiv):
    fixed = indiv[:]
    for x in range(len(fixed)):
        for y in range(x+1, len(fixed)):
            if fixed[x] == fixed[y]:
                for _ in range(10):
                    new_val = (random.randint(0,2), random.randint(0,3))
                    if new_val not in fixed:
                        fixed[y] = new_val
                        break
                else:
                    fixed[y] = (random.randint(0,2), random.randint(0,3))
    return fixed

def apply_mutation(indiv, prob):
    return [(random.randint(0,2), random.randint(0,3))
            if random.random() < prob else g
            for g in indiv]

parent1 = [(0,0),(1,1),(2,2),(0,1),(1,2),(2,3)]
parent2 = [(0,0),(1,1),(2,2),(0,1),(1,2),(2,3)]

offspring1, offspring2 = mix_parents(parent1, parent2, 3)

print("Parent 1:", parent1)
print("Parent 2:", parent2)
print("Child 1:", offspring1)
print("Child 2:", offspring2)
print("Conflicts in Child 1:", calc_conflicts(offspring1))

dup_chrom = [(0,0),(0,0),(1,1),(2,2),(1,1),(2,3)]
print("\nBefore Repair:", dup_chrom)
print("Conflicts:", calc_conflicts(dup_chrom))

fixed_chrom = fix_duplicates(dup_chrom)
print("After Repair:", fixed_chrom)
print("Conflicts:", calc_conflicts(fixed_chrom))

Parent 1: [(0, 0), (1, 1), (2, 2), (0, 1), (1, 2), (2, 3)]
Parent 2: [(0, 0), (1, 1), (2, 2), (0, 1), (1, 2), (2, 3)]
Child 1: [(0, 0), (1, 1), (2, 2), (0, 1), (1, 2), (2, 3)]
Child 2: [(0, 0), (1, 1), (2, 2), (0, 1), (1, 2), (2, 3)]
Conflicts in Child 1: 0

Before Repair: [(0, 0), (0, 0), (1, 1), (2, 2), (1, 1), (2, 3)]
Conflicts: 2
After Repair: [(0, 0), (1, 2), (1, 1), (2, 2), (1, 0), (2, 3)]
Conflicts: 0


In [8]:
#part c
def select_tournament(group):
    ind1, ind2 = random.sample(group, 2)
    return ind1 if compute_score(ind1) > compute_score(ind2) else ind2

def execute_schedule_ga(size, gens, mut_rate):
    pop = [create_individual() for _ in range(size)]
    best_indiv = None
    min_conf = float('inf')
    best_each_gen = []

    for g in range(gens):
        fit_vals = [compute_score(ind) for ind in pop]
        conf_vals = [calc_conflicts(ind) for ind in pop]

        top_idx = fit_vals.index(max(fit_vals))
        top_conf = conf_vals[top_idx]

        best_each_gen.append(fit_vals[top_idx])

        if top_conf < min_conf:
            min_conf = top_conf
            best_indiv = pop[top_idx]

        if top_conf == 0:
            print(f"\nSolution found at generation {g}: {pop[top_idx]}")

        next_gen = []

        while len(next_gen) < size:
            par1 = select_tournament(pop)
            par2 = select_tournament(pop)

            cut_pt = random.randint(1, 5)
            child1, child2 = mix_parents(par1, par2, cut_pt)

            child1 = fix_duplicates(apply_mutation(child1, mut_rate))
            child2 = fix_duplicates(apply_mutation(child2, mut_rate))

            next_gen += [child1, child2][:size - len(next_gen)]

        pop = next_gen

    print("\nBest Schedule Found:", best_indiv)
    print("Conflicts:", calc_conflicts(best_indiv))

    return best_each_gen


execute_schedule_ga(size=20, gens=50, mut_rate=0.1)


Solution found at generation 0: [(2, 2), (1, 2), (2, 0), (0, 1), (1, 1), (0, 3)]

Solution found at generation 1: [(2, 1), (1, 0), (2, 0), (0, 1), (1, 3), (2, 3)]

Solution found at generation 2: [(2, 1), (1, 2), (1, 3), (2, 2), (0, 2), (1, 0)]

Solution found at generation 3: [(1, 3), (2, 2), (0, 0), (1, 2), (2, 0), (1, 1)]

Solution found at generation 4: [(2, 1), (0, 0), (1, 2), (2, 2), (2, 3), (0, 2)]

Solution found at generation 5: [(2, 0), (0, 0), (1, 2), (2, 2), (2, 1), (1, 1)]

Solution found at generation 6: [(0, 3), (1, 3), (2, 0), (2, 2), (0, 1), (0, 2)]

Solution found at generation 7: [(2, 0), (0, 0), (1, 2), (2, 2), (0, 1), (1, 0)]

Solution found at generation 8: [(0, 3), (1, 3), (1, 2), (1, 1), (2, 3), (2, 2)]

Solution found at generation 9: [(2, 1), (1, 1), (2, 3), (0, 1), (1, 0), (2, 2)]

Solution found at generation 10: [(2, 1), (1, 3), (1, 2), (0, 2), (1, 1), (0, 3)]

Solution found at generation 11: [(2, 1), (0, 0), (1, 2), (0, 2), (0, 1), (2, 0)]

Solution foun

[100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100]